<p style="margin: 5px 0 0 0; color: #666;"><em>Desarrollado con Claude - Anthropic</em></p>


# 9. Excel para Análisis de Datos

*Desarrollado con Claude - Anthropic*

In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, PieChart, LineChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows
import xlsxwriter
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías cargadas correctamente")
print(f"Pandas versión: {pd.__version__}")
print(f"OpenPyXL versión: {openpyxl.__version__}")

✓ Librerías cargadas correctamente
Pandas versión: 3.0.0
OpenPyXL versión: 3.1.5


## Preparación: Crear Datos de Ejemplo

### Que es?
Esta seccion crea datasets de practica para simular escenarios tipicos de analisis en Excel.

### Para que sirve?
Trabajar con datos controlados permite enfocarse en formulas, tablas y reportes sin depender de fuentes externas.

### Como se usa?
1. Generar tablas base de ventas y referencia
2. Verificar estructura y tipos de columnas
3. Reutilizar estos datos en las secciones siguientes

In [2]:
print("=" * 70)
print("CREACIÓN DE DATASETS DE EJEMPLO".center(70))
print("=" * 70)

# Dataset de ventas
np.random.seed(42)
fechas = pd.date_range('2023-01-01', periods=100, freq='D')

df_ventas = pd.DataFrame({
    'Fecha': fechas,
    'Vendedor': np.random.choice(['Ana', 'Carlos', 'María', 'Luis', 'Pedro'], 100),
    'Producto': np.random.choice(['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares'], 100),
    'Región': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], 100),
    'Cantidad': np.random.randint(1, 10, 100),
    'Precio_Unitario': np.random.uniform(20, 800, 100).round(2),
    'Descuento': np.random.uniform(0, 0.2, 100).round(2)
})

df_ventas['Total'] = (df_ventas['Cantidad'] * df_ventas['Precio_Unitario'] * 
                      (1 - df_ventas['Descuento'])).round(2)
df_ventas['Mes'] = df_ventas['Fecha'].dt.month_name()
df_ventas['Trimestre'] = 'Q' + df_ventas['Fecha'].dt.quarter.astype(str)

print(f"\n✓ Dataset de ventas creado: {df_ventas.shape}")
print(df_ventas.head())

# Dataset de productos (tabla de referencia)
df_productos = pd.DataFrame({
    'Producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares'],
    'Categoría': ['Computadoras', 'Accesorios', 'Accesorios', 'Computadoras', 'Audio'],
    'Precio_Lista': [899.99, 29.99, 79.99, 299.99, 59.99],
    'Stock': [50, 200, 150, 75, 120],
    'Proveedor': ['HP', 'Logitech', 'Logitech', 'Samsung', 'Sony']
})

print(f"\n✓ Dataset de productos creado: {df_productos.shape}")
print(df_productos)

                   CREACIÓN DE DATASETS DE EJEMPLO                    

✓ Dataset de ventas creado: (100, 10)
       Fecha Vendedor Producto Región  Cantidad  Precio_Unitario  Descuento  \
0 2023-01-01     Luis  Monitor    Sur         3           517.20       0.17   
1 2023-01-02    Pedro   Laptop    Sur         9           586.35       0.06   
2 2023-01-03    María  Monitor    Sur         4           781.16       0.18   
3 2023-01-04    Pedro    Mouse   Este         1           422.71       0.08   
4 2023-01-05    Pedro   Laptop   Este         4           271.91       0.00   

     Total      Mes Trimestre  
0  1287.83  January        Q1  
1  4960.52  January        Q1  
2  2562.20  January        Q1  
3   388.89  January        Q1  
4  1087.64  January        Q1  

✓ Dataset de productos creado: (5, 5)
      Producto     Categoría  Precio_Lista  Stock Proveedor
0       Laptop  Computadoras        899.99     50        HP
1        Mouse    Accesorios         29.99    200  Logitech
2   

## Funciones Avanzadas de Excel en Python

### BUSCARV (VLOOKUP), SI (IF), SUMAR.SI (SUMIF), CONTAR.SI (COUNTIF)

### Que es?
Son equivalencias entre funciones clasicas de Excel y operaciones en Pandas/Numpy.

### Para que sirve?
Facilita migrar habilidades de Excel a Python sin perder logica de negocio.

### Como se usa?
1. Identificar la funcion de Excel
2. Aplicar su equivalente en Python
3. Comparar resultados para validar comprension

In [3]:
print("=" * 70)
print("FUNCIONES AVANZADAS - BUSCARV (VLOOKUP)".center(70))
print("=" * 70)

# BUSCARV en Excel: busca un valor y devuelve otro de la misma fila
# En Python: usando merge() o map()

print("\n1. BUSCARV - Agregar categoría a ventas")
print("-" * 70)
print("Excel: =BUSCARV(Producto, TablaProductos, 2, FALSO)")
print("Python: merge() o map()\n")

# Método 1: Con merge (equivalente a BUSCARV)
df_ventas_enriquecido = df_ventas.merge(
    df_productos[['Producto', 'Categoría', 'Proveedor']], 
    on='Producto', 
    how='left'
)

print("Resultado con merge (BUSCARV):")
print(df_ventas_enriquecido[['Producto', 'Categoría', 'Proveedor', 'Total']].head())

# Método 2: Con map (más simple para una columna)
dict_categorias = df_productos.set_index('Producto')['Categoría'].to_dict()
df_ventas['Categoría'] = df_ventas['Producto'].map(dict_categorias)

print("\nResultado con map:")
print(df_ventas[['Producto', 'Categoría']].head())

print("\n💡 BUSCARV en Excel vs Python:")
print("  Excel: =BUSCARV(valor_buscado, rango, columna, [coincidencia])")
print("  Python: df.merge() o Series.map()")

               FUNCIONES AVANZADAS - BUSCARV (VLOOKUP)                

1. BUSCARV - Agregar categoría a ventas
----------------------------------------------------------------------
Excel: =BUSCARV(Producto, TablaProductos, 2, FALSO)
Python: merge() o map()

Resultado con merge (BUSCARV):
  Producto     Categoría Proveedor    Total
0  Monitor  Computadoras   Samsung  1287.83
1   Laptop  Computadoras        HP  4960.52
2  Monitor  Computadoras   Samsung  2562.20
3    Mouse    Accesorios  Logitech   388.89
4   Laptop  Computadoras        HP  1087.64

Resultado con map:
  Producto     Categoría
0  Monitor  Computadoras
1   Laptop  Computadoras
2  Monitor  Computadoras
3    Mouse    Accesorios
4   Laptop  Computadoras

💡 BUSCARV en Excel vs Python:
  Excel: =BUSCARV(valor_buscado, rango, columna, [coincidencia])
  Python: df.merge() o Series.map()


In [4]:
print("=" * 70)
print("FUNCIONES AVANZADAS - SI (IF)".center(70))
print("=" * 70)

# SI en Excel: devuelve un valor si la condición es verdadera
# En Python: usando np.where() o apply()

print("\n1. SI simple - Clasificar ventas")
print("-" * 70)
print("Excel: =SI(Total > 500, 'Alta', 'Baja')")
print("Python: np.where()\n")

df_ventas['Clasificación'] = np.where(df_ventas['Total'] > 500, 'Alta', 'Baja')

print("Resultado:")
print(df_ventas[['Total', 'Clasificación']].head(10))

# SI anidado múltiple
print("\n2. SI anidado - Clasificación múltiple")
print("-" * 70)
print("Excel: =SI(Total > 1000, 'Premium', SI(Total > 500, 'Alta', 'Normal'))")
print("Python: np.select()\n")

condiciones = [
    df_ventas['Total'] > 1000,
    df_ventas['Total'] > 500,
    df_ventas['Total'] > 200
]
valores = ['Premium', 'Alta', 'Media']
df_ventas['Segmento'] = np.select(condiciones, valores, default='Baja')

print("Resultado:")
print(df_ventas[['Total', 'Segmento']].head(10))
print(f"\nDistribución por segmento:\n{df_ventas['Segmento'].value_counts()}")

# SI con texto
print("\n3. SI con condiciones de texto")
print("-" * 70)
df_ventas['Zona'] = np.where(
    df_ventas['Región'].isin(['Norte', 'Sur']), 
    'Nacional', 
    'Internacional'
)

print(df_ventas[['Región', 'Zona']].head(10))

print("\n💡 SI en Excel vs Python:")
print("  Excel: =SI(condición, valor_si_verdadero, valor_si_falso)")
print("  Python: np.where() para condiciones simples")
print("  Python: np.select() para múltiples condiciones")

                    FUNCIONES AVANZADAS - SI (IF)                     

1. SI simple - Clasificar ventas
----------------------------------------------------------------------
Excel: =SI(Total > 500, 'Alta', 'Baja')
Python: np.where()

Resultado:
     Total Clasificación
0  1287.83          Alta
1  4960.52          Alta
2  2562.20          Alta
3   388.89          Baja
4  1087.64          Alta
5   525.00          Alta
6  1133.12          Alta
7  1362.62          Alta
8   526.18          Alta
9   257.71          Baja

2. SI anidado - Clasificación múltiple
----------------------------------------------------------------------
Excel: =SI(Total > 1000, 'Premium', SI(Total > 500, 'Alta', 'Normal'))
Python: np.select()

Resultado:
     Total Segmento
0  1287.83  Premium
1  4960.52  Premium
2  2562.20  Premium
3   388.89    Media
4  1087.64  Premium
5   525.00     Alta
6  1133.12  Premium
7  1362.62  Premium
8   526.18     Alta
9   257.71    Media

Distribución por segmento:
Segmento
Premium

In [5]:
print("=" * 70)
print("FUNCIONES AVANZADAS - SUMAR.SI Y CONTAR.SI".center(70))
print("=" * 70)

# SUMAR.SI: suma con condición
print("\n1. SUMAR.SI (SUMIF) - Suma condicional")
print("-" * 70)
print("Excel: =SUMAR.SI(rango_criterio, criterio, rango_suma)")
print("Python: groupby() + sum() o query()\n")

# Total de ventas por región
ventas_por_region = df_ventas.groupby('Región')['Total'].sum()
print("Total de ventas por región:")
print(ventas_por_region)

# Ventas de un producto específico
ventas_laptop = df_ventas[df_ventas['Producto'] == 'Laptop']['Total'].sum()
print(f"\nVentas totales de Laptop: ${ventas_laptop:,.2f}")

# SUMAR.SI.CONJUNTO (múltiples condiciones)
print("\n2. SUMAR.SI.CONJUNTO - Múltiples condiciones")
print("-" * 70)
print("Excel: =SUMAR.SI.CONJUNTO(rango_suma, criterio1, rango1, criterio2, rango2)")
print("Python: query() o filtrado múltiple\n")

# Ventas de Laptop en región Norte
ventas_laptop_norte = df_ventas[
    (df_ventas['Producto'] == 'Laptop') & 
    (df_ventas['Región'] == 'Norte')
]['Total'].sum()
print(f"Ventas de Laptop en Norte: ${ventas_laptop_norte:,.2f}")

# CONTAR.SI: contar con condición
print("\n3. CONTAR.SI (COUNTIF) - Contar condicional")
print("-" * 70)
print("Excel: =CONTAR.SI(rango, criterio)")
print("Python: value_counts() o len() con filtro\n")

# Contar ventas por producto
conteo_por_producto = df_ventas['Producto'].value_counts()
print("Número de ventas por producto:")
print(conteo_por_producto)

# Contar ventas superiores a 500
ventas_altas = (df_ventas['Total'] > 500).sum()
print(f"\nNúmero de ventas > $500: {ventas_altas}")

# CONTAR.SI.CONJUNTO
print("\n4. CONTAR.SI.CONJUNTO - Múltiples condiciones")
print("-" * 70)
ventas_premium = len(df_ventas[
    (df_ventas['Total'] > 1000) & 
    (df_ventas['Región'] == 'Norte')
])
print(f"Ventas Premium en Norte: {ventas_premium}")

# Crear tabla resumen
print("\n5. Tabla resumen completa")
print("-" * 70)
resumen = df_ventas.groupby('Producto').agg({
    'Total': ['sum', 'mean', 'count'],
    'Cantidad': 'sum'
}).round(2)
resumen.columns = ['Total_Ventas', 'Promedio', 'Num_Transacciones', 'Unidades']
print(resumen)

print("\n💡 Funciones condicionales Excel vs Python:")
print("  SUMAR.SI → df[condición]['columna'].sum()")
print("  CONTAR.SI → (df['columna'] == valor).sum()")
print("  PROMEDIO.SI → df[condición]['columna'].mean()")
print("  MAX.SI, MIN.SI → df[condición]['columna'].max/min()")

              FUNCIONES AVANZADAS - SUMAR.SI Y CONTAR.SI              

1. SUMAR.SI (SUMIF) - Suma condicional
----------------------------------------------------------------------
Excel: =SUMAR.SI(rango_criterio, criterio, rango_suma)
Python: groupby() + sum() o query()

Total de ventas por región:
Región
Este     40260.87
Norte    33435.75
Oeste    54256.20
Sur      40366.16
Name: Total, dtype: float64

Ventas totales de Laptop: $49,983.95

2. SUMAR.SI.CONJUNTO - Múltiples condiciones
----------------------------------------------------------------------
Excel: =SUMAR.SI.CONJUNTO(rango_suma, criterio1, rango1, criterio2, rango2)
Python: query() o filtrado múltiple

Ventas de Laptop en Norte: $5,121.62

3. CONTAR.SI (COUNTIF) - Contar condicional
----------------------------------------------------------------------
Excel: =CONTAR.SI(rango, criterio)
Python: value_counts() o len() con filtro

Número de ventas por producto:
Producto
Laptop         25
Monitor        23
Teclado        2

## Tablas Dinámicas

Creación de tablas dinámicas con Pandas.

### Que es?
Una tabla dinamica resume grandes volumenes de datos mediante agrupaciones y agregaciones.

### Para que sirve?
Permite responder preguntas de negocio rapidamente: totales por region, producto, vendedor o periodo.

### Como se usa?
1. Elegir dimensiones (filas/columnas)
2. Definir metrica agregada (suma, promedio, conteo)
3. Analizar patrones y comparaciones

In [6]:
print("=" * 70)
print("TABLAS DINÁMICAS (PIVOT TABLES)".center(70))
print("=" * 70)

# 1. Tabla dinámica simple
print("\n1. TABLA DINÁMICA SIMPLE - Ventas por Producto y Región")
print("-" * 70)

pivot_simple = pd.pivot_table(
    df_ventas,
    values='Total',
    index='Producto',
    columns='Región',
    aggfunc='sum',
    fill_value=0
)

print(pivot_simple.round(2))

# 2. Tabla dinámica con múltiples agregaciones
print("\n2. TABLA DINÁMICA - Múltiples funciones")
print("-" * 70)

pivot_multi = pd.pivot_table(
    df_ventas,
    values='Total',
    index='Producto',
    columns='Región',
    aggfunc=['sum', 'mean', 'count'],
    fill_value=0
)

print(pivot_multi.round(2))

# 3. Tabla dinámica con totales
print("\n3. TABLA DINÁMICA - Con totales (margins)")
print("-" * 70)

pivot_totales = pd.pivot_table(
    df_ventas,
    values='Total',
    index='Producto',
    columns='Región',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total General'
)

print(pivot_totales.round(2))

# 4. Tabla dinámica multidimensional
print("\n4. TABLA DINÁMICA - Múltiples dimensiones")
print("-" * 70)

pivot_multi_dim = pd.pivot_table(
    df_ventas,
    values=['Total', 'Cantidad'],
    index=['Trimestre', 'Producto'],
    columns='Región',
    aggfunc={'Total': 'sum', 'Cantidad': 'sum'},
    fill_value=0
)

print(pivot_multi_dim.round(2))

# 5. Tabla dinámica por vendedor
print("\n5. TABLA DINÁMICA - Análisis por vendedor")
print("-" * 70)

pivot_vendedor = pd.pivot_table(
    df_ventas,
    values='Total',
    index='Vendedor',
    columns='Trimestre',
    aggfunc=['sum', 'count'],
    fill_value=0,
    margins=True
)

print(pivot_vendedor.round(2))

# 6. Tabla cruzada (crosstab)
print("\n6. TABLA CRUZADA - Frecuencias")
print("-" * 70)

crosstab = pd.crosstab(
    df_ventas['Producto'], 
    df_ventas['Región'],
    margins=True,
    margins_name='Total'
)

print(crosstab)

# Porcentajes
print("\nPorcentajes por fila:")
crosstab_pct = pd.crosstab(
    df_ventas['Producto'], 
    df_ventas['Región'],
    normalize='index'
) * 100
print(crosstab_pct.round(2))

print("\n💡 TABLAS DINÁMICAS:")
print("  • pivot_table(): para agregaciones numéricas")
print("  • crosstab(): para tablas de frecuencia")
print("  • index: filas de la tabla")
print("  • columns: columnas de la tabla")
print("  • values: valores a agregar")
print("  • aggfunc: función de agregación (sum, mean, count, etc.)")
print("  • margins: agregar totales")

                   TABLAS DINÁMICAS (PIVOT TABLES)                    

1. TABLA DINÁMICA SIMPLE - Ventas por Producto y Región
----------------------------------------------------------------------
Región           Este     Norte     Oeste       Sur
Producto                                           
Auriculares   5543.16   1812.53   6123.17   6511.79
Laptop       15333.04   5121.62  17001.00  12528.29
Monitor       8350.69  10309.30   8139.97  13093.46
Mouse         3534.40   9689.20  10481.90   2781.85
Teclado       7499.58   6503.10  12510.16   5450.77

2. TABLA DINÁMICA - Múltiples funciones
----------------------------------------------------------------------
                  sum                                   mean           \
Región           Este     Norte     Oeste       Sur     Este    Norte   
Producto                                                                
Auriculares   5543.16   1812.53   6123.17   6511.79  1385.79   604.18   
Laptop       15333.04   5121.62  

## Gráficos y Dashboards

Crear archivos Excel con gráficos embebidos.

### Que es?
Esta parte muestra como construir reportes visuales y dashboards exportables en archivos Excel.

### Para que sirve?
Automatiza entregables ejecutivos sin trabajo manual repetitivo.

### Como se usa?
1. Crear hoja resumen con KPIs
2. Insertar graficos y aplicar estilos
3. Guardar archivo listo para compartir

In [7]:
print("=" * 70)
print("CREAR EXCEL CON GRÁFICOS".center(70))
print("=" * 70)

# Crear archivo Excel con gráficos usando openpyxl
print("\nCreando archivo Excel con gráficos...")

wb = Workbook()
ws = wb.active
ws.title = "Dashboard Ventas"

# Agregar datos de resumen
resumen_region = df_ventas.groupby('Región')['Total'].sum().reset_index()
resumen_region.columns = ['Región', 'Ventas']

# Escribir encabezados con estilo
ws['A1'] = 'Región'
ws['B1'] = 'Ventas Totales'

# Estilo para encabezados
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True, size=12)

for cell in ['A1', 'B1']:
    ws[cell].fill = header_fill
    ws[cell].font = header_font
    ws[cell].alignment = Alignment(horizontal='center')

# Escribir datos
for idx, row in resumen_region.iterrows():
    ws.cell(row=idx+2, column=1, value=row['Región'])
    ws.cell(row=idx+2, column=2, value=row['Ventas'])

# Crear gráfico de barras
chart = BarChart()
chart.title = "Ventas por Región"
chart.style = 10
chart.x_axis.title = "Región"
chart.y_axis.title = "Ventas ($)"

# Definir rango de datos
data = Reference(ws, min_col=2, min_row=1, max_row=len(resumen_region)+1)
cats = Reference(ws, min_col=1, min_row=2, max_row=len(resumen_region)+1)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.height = 10
chart.width = 20

ws.add_chart(chart, "D2")

# Agregar hoja con datos detallados
ws2 = wb.create_sheet("Datos Detallados")

# Escribir DataFrame a Excel
for r_idx, row in enumerate(dataframe_to_rows(df_ventas.head(20), index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        cell = ws2.cell(row=r_idx, column=c_idx, value=value)
        
        # Estilo para encabezado
        if r_idx == 1:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')

# Ajustar ancho de columnas
for col in ws2.columns:
    max_length = 0
    column = col[0].column_letter
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 20)
    ws2.column_dimensions[column].width = adjusted_width

# Guardar archivo
wb.save('dashboard_ventas.xlsx')
print("✓ Archivo 'dashboard_ventas.xlsx' creado con gráficos")

print("\n📊 Contenido del archivo:")
print("  • Hoja 1: Dashboard con gráfico de barras")
print("  • Hoja 2: Datos detallados con formato")

                       CREAR EXCEL CON GRÁFICOS                       

Creando archivo Excel con gráficos...
✓ Archivo 'dashboard_ventas.xlsx' creado con gráficos

📊 Contenido del archivo:
  • Hoja 1: Dashboard con gráfico de barras
  • Hoja 2: Datos detallados con formato


In [9]:
print("=" * 70)
print("CREAR DASHBOARD COMPLETO CON XLSXWRITER".center(70))
print("=" * 70)

# XlsxWriter ofrece más opciones para gráficos y formato
print("\nCreando dashboard completo...")

workbook = xlsxwriter.Workbook('dashboard_completo.xlsx')
worksheet = workbook.add_worksheet('Dashboard')

# Formatos
title_format = workbook.add_format({
    'bold': True,
    'font_size': 14,
    'bg_color': '#366092',
    'font_color': 'white',
    'align': 'center'
})

header_format = workbook.add_format({
    'bold': True,
    'bg_color': '#D9E1F2',
    'border': 1
})

number_format = workbook.add_format({'num_format': '$#,##0.00'})
percent_format = workbook.add_format({'num_format': '0.00%'})

# Título
worksheet.merge_range('A1:F1', 'DASHBOARD DE VENTAS 2023', title_format)

# KPIs en la parte superior
row = 2
kpis = [
    ('Ventas Totales', df_ventas['Total'].sum()),
    ('Promedio por Venta', df_ventas['Total'].mean()),
    ('Número de Ventas', len(df_ventas))
]

col = 0
for kpi_name, kpi_value in kpis:
    worksheet.write(row, col, kpi_name, header_format)
    if col < 2:
        worksheet.write(row + 1, col, kpi_value, number_format)
    else:
        worksheet.write(row + 1, col, kpi_value)
    col += 2

# Tabla de ventas por producto
row = 5
worksheet.write(row, 0, 'Producto', header_format)
worksheet.write(row, 1, 'Ventas', header_format)
worksheet.write(row, 2, 'Unidades', header_format)

ventas_producto = df_ventas.groupby('Producto').agg({
    'Total': 'sum',
    'Cantidad': 'sum'
}).sort_values('Total', ascending=False)

row += 1
for producto, datos in ventas_producto.iterrows():
    worksheet.write(row, 0, producto)
    worksheet.write(row, 1, datos['Total'], number_format)
    worksheet.write(row, 2, int(datos['Cantidad']))
    row += 1

# Crear gráfico de columnas
chart1 = workbook.add_chart({'type': 'column'})
chart1.add_series({
    'name': 'Ventas por Producto',
    'categories': f'=Dashboard!$A$7:$A${6+len(ventas_producto)}',
    'values': f'=Dashboard!$B$7:$B${6+len(ventas_producto)}',
    'fill': {'color': '#4472C4'},
})

chart1.set_title({'name': 'Ventas por Producto'})
chart1.set_x_axis({'name': 'Producto'})
chart1.set_y_axis({'name': 'Ventas ($)'})
chart1.set_style(10)

worksheet.insert_chart('E6', chart1, {'x_scale': 1.5, 'y_scale': 1.5})

# Gráfico circular para regiones
row_region = row + 2
worksheet.write(row_region, 0, 'Región', header_format)
worksheet.write(row_region, 1, 'Ventas', header_format)

ventas_region = df_ventas.groupby('Región')['Total'].sum().sort_values(ascending=False)

row_region += 1
for region, ventas in ventas_region.items():
    worksheet.write(row_region, 0, region)
    worksheet.write(row_region, 1, ventas, number_format)
    row_region += 1

# Gráfico de pastel
chart2 = workbook.add_chart({'type': 'pie'})
chart2.add_series({
    'name': 'Ventas por Región',
    'categories': f'=Dashboard!$A${row+4}:$A${row+3+len(ventas_region)}',
    'values': f'=Dashboard!$B${row+4}:$B${row+3+len(ventas_region)}',
    'data_labels': {'percentage': True},
})

chart2.set_title({'name': 'Distribución por Región'})
chart2.set_style(10)

worksheet.insert_chart(f'E{row+2}', chart2, {'x_scale': 1.5, 'y_scale': 1.5})

# Hoja con datos completos
worksheet2 = workbook.add_worksheet('Datos')

# Escribir DataFrame usando XlsxWriter (no pandas.to_excel con Workbook)
for col_idx, col_name in enumerate(df_ventas.columns):
    worksheet2.write(0, col_idx, col_name, header_format)

for row_idx, row_data in enumerate(df_ventas.itertuples(index=False), start=1):
    for col_idx, value in enumerate(row_data):
        worksheet2.write(row_idx, col_idx, value)

worksheet2.set_column(0, len(df_ventas.columns) - 1, 14)

workbook.close()

print("✓ Archivo 'dashboard_completo.xlsx' creado")
print("\n📊 Contenido del dashboard:")
print("  • KPIs principales")
print("  • Tabla y gráfico de barras por producto")
print("  • Tabla y gráfico circular por región")
print("  • Hoja con datos completos")

print("\n💡 XLSXWRITER vs OPENPYXL:")
print("  XlsxWriter: mejor para crear archivos nuevos con muchos gráficos")
print("  OpenPyXL: mejor para leer/modificar archivos existentes")

               CREAR DASHBOARD COMPLETO CON XLSXWRITER                

Creando dashboard completo...
✓ Archivo 'dashboard_completo.xlsx' creado

📊 Contenido del dashboard:
  • KPIs principales
  • Tabla y gráfico de barras por producto
  • Tabla y gráfico circular por región
  • Hoja con datos completos

💡 XLSXWRITER vs OPENPYXL:
  XlsxWriter: mejor para crear archivos nuevos con muchos gráficos
  OpenPyXL: mejor para leer/modificar archivos existentes


## Power Query (Transformación de Datos)

Simulación de operaciones de Power Query con Pandas.

### Que es?
Power Query es una herramienta de transformacion y limpieza de datos previa al analisis.

### Para que sirve?
Estandariza procesos ETL ligeros: limpiar, combinar y remodelar datos de forma reproducible.

### Como se usa?
1. Importar datos
2. Aplicar transformaciones paso a paso
3. Cargar resultado final para analisis

In [11]:
print("=" * 70)
print("POWER QUERY - TRANSFORMACIONES DE DATOS".center(70))
print("=" * 70)

# Power Query en Excel: herramienta para importar y transformar datos
# En Python: Pandas ofrece operaciones similares

print("\n1. IMPORTAR Y CONECTAR DATOS")
print("-" * 70)
print("Power Query: Obtener datos → Excel/CSV/Base de datos")
print("Python: pd.read_excel(), pd.read_csv()\n")

# Guardar y recargar
df_ventas.to_excel('ventas_raw.xlsx', index=False)
df_importado = pd.read_excel('ventas_raw.xlsx')
print(f"✓ Datos importados: {df_importado.shape}")

print("\n2. QUITAR COLUMNAS")
print("-" * 70)
print("Power Query: Quitar columnas")
print("Python: drop()\n")

df_limpio = df_importado.drop(['Descuento'], axis=1)
print(f"Columnas después de eliminar: {list(df_limpio.columns)}")

print("\n3. FILTRAR FILAS")
print("-" * 70)
print("Power Query: Filtrar filas")
print("Python: query() o filtrado booleano\n")

df_filtrado = df_limpio[df_limpio['Total'] > 200]
print(f"Filas después de filtrar (Total > 200): {len(df_filtrado)}")

print("\n4. AGREGAR COLUMNA PERSONALIZADA")
print("-" * 70)
print("Power Query: Agregar columna personalizada")
print("Python: assign() o crear columna directamente\n")

df_custom = df_filtrado.copy()
df_custom['Comisión'] = df_custom['Total'] * 0.05
df_custom['Año'] = df_custom['Fecha'].dt.year
df_custom['Mes_Num'] = df_custom['Fecha'].dt.month

print("Nuevas columnas creadas:")
print(df_custom[['Total', 'Comisión', 'Año', 'Mes_Num']].head())

print("\n5. AGRUPAR FILAS")
print("-" * 70)
print("Power Query: Agrupar por")
print("Python: groupby()\n")

df_agrupado = df_custom.groupby('Vendedor').agg({
    'Total': ['sum', 'mean', 'count'],
    'Comisión': 'sum'
}).round(2)

df_agrupado.columns = ['Total_Ventas', 'Promedio_Venta', 'Num_Ventas', 'Comision_Total']
print(df_agrupado)

print("\n6. COMBINAR CONSULTAS (MERGE)")
print("-" * 70)
print("Power Query: Combinar consultas")
print("Python: merge()\n")

# Evitar columnas duplicadas como Categoría_x/Categoría_y
df_custom_merge = df_custom.drop(columns=['Categoría'], errors='ignore')

df_combinado = df_custom_merge.merge(
    df_productos[['Producto', 'Categoría', 'Precio_Lista']],
    on='Producto',
    how='left'
)

print("Datos combinados:")
print(df_combinado[['Producto', 'Categoría', 'Precio_Lista', 'Total']].head())

print("\n7. TRANSPONER")
print("-" * 70)
print("Power Query: Transponer")
print("Python: transpose() o T\n")

# Ejemplo con tabla pequeña
resumen_vendedor = df_custom.groupby('Vendedor')['Total'].sum().head(3)
print("Original:")
print(resumen_vendedor)
print("\nTranspuesto:")
print(resumen_vendedor.to_frame().T)

print("\n8. DINAMIZAR/ANULAR DINAMIZACIÓN")
print("-" * 70)
print("Power Query: Dinamizar columnas")
print("Python: pivot() / melt()\n")

# Dinamizar (pivot)
df_pivot = df_custom.pivot_table(
    values='Total',
    index='Vendedor',
    columns='Región',
    aggfunc='sum',
    fill_value=0
).head(3)

print("Tabla dinamizada:")
print(df_pivot)

# Anular dinamización (melt)
df_unpivot = df_pivot.reset_index().melt(
    id_vars='Vendedor',
    var_name='Región',
    value_name='Total'
)

print("\nTabla sin dinamizar (formato largo):")
print(df_unpivot.head())

print("\n9. DIVIDIR COLUMNA")
print("-" * 70)
print("Power Query: Dividir columna por delimitador")
print("Python: str.split()\n")

# Crear columna con datos para dividir
df_ejemplo = pd.DataFrame({
    'Nombre_Completo': ['Juan Pérez', 'María García', 'Carlos López']
})

df_ejemplo[['Nombre', 'Apellido']] = df_ejemplo['Nombre_Completo'].str.split(' ', expand=True)
print(df_ejemplo)

print("\n10. REEMPLAZAR VALORES")
print("-" * 70)
print("Power Query: Reemplazar valores")
print("Python: replace()\n")

df_reemplazo = df_custom.copy()
df_reemplazo['Región'] = df_reemplazo['Región'].replace({
    'Norte': 'N',
    'Sur': 'S',
    'Este': 'E',
    'Oeste': 'O'
})

print(df_reemplazo[['Región']].head())

print("\n💡 POWER QUERY vs PANDAS:")
print("  Ambos permiten transformaciones sin escribir código (Power Query tiene UI)")
print("  Pandas es más potente y flexible para transformaciones complejas")
print("  Power Query es ideal para usuarios de negocio sin programación")

               POWER QUERY - TRANSFORMACIONES DE DATOS                

1. IMPORTAR Y CONECTAR DATOS
----------------------------------------------------------------------
Power Query: Obtener datos → Excel/CSV/Base de datos
Python: pd.read_excel(), pd.read_csv()

✓ Datos importados: (100, 14)

2. QUITAR COLUMNAS
----------------------------------------------------------------------
Power Query: Quitar columnas
Python: drop()

Columnas después de eliminar: ['Fecha', 'Vendedor', 'Producto', 'Región', 'Cantidad', 'Precio_Unitario', 'Total', 'Mes', 'Trimestre', 'Categoría', 'Clasificación', 'Segmento', 'Zona']

3. FILTRAR FILAS
----------------------------------------------------------------------
Power Query: Filtrar filas
Python: query() o filtrado booleano

Filas después de filtrar (Total > 200): 93

4. AGREGAR COLUMNA PERSONALIZADA
----------------------------------------------------------------------
Power Query: Agregar columna personalizada
Python: assign() o crear columna directam

## Fórmulas Matriciales

Operaciones avanzadas con arrays.

### Que es?
Las formulas matriciales operan sobre rangos completos en lugar de celda por celda.

### Para que sirve?
Reducen errores manuales y aceleran calculos complejos en analisis financiero o comercial.

### Como se usa?
1. Definir arreglos o columnas objetivo
2. Aplicar operaciones vectorizadas
3. Validar consistencia de resultados

In [12]:
print("=" * 70)
print("FÓRMULAS MATRICIALES".center(70))
print("=" * 70)

# Fórmulas matriciales en Excel: operaciones sobre rangos de celdas
# En Python: operaciones vectorizadas con NumPy/Pandas

print("\n1. SUMA DE PRODUCTOS (SUMPRODUCT)")
print("-" * 70)
print("Excel: =SUMAPRODUCTO(rango1, rango2)")
print("Python: (serie1 * serie2).sum()\n")

# Calcular ventas totales
cantidad = df_ventas['Cantidad'].values
precio = df_ventas['Precio_Unitario'].values
descuento_factor = (1 - df_ventas['Descuento']).values

total_calculado = (cantidad * precio * descuento_factor).sum()
print(f"Total de ventas (SUMAPRODUCTO): ${total_calculado:,.2f}")
print(f"Verificación con columna Total: ${df_ventas['Total'].sum():,.2f}")

print("\n2. OPERACIONES MATRICIALES ELEMENTO A ELEMENTO")
print("-" * 70)
print("Excel: Fórmula matricial con Ctrl+Shift+Enter")
print("Python: operaciones vectorizadas\n")

# Crear vectores
precios = np.array([100, 200, 150, 300])
cantidades = np.array([2, 3, 1, 4])
descuentos = np.array([0.1, 0.15, 0.05, 0.2])

# Operación vectorizada
totales = precios * cantidades * (1 - descuentos)

print("Precios:", precios)
print("Cantidades:", cantidades)
print("Descuentos:", descuentos)
print("Totales:", totales)
print(f"Suma total: ${totales.sum():,.2f}")

print("\n3. FILTRAR CON CONDICIONES (FILTER en Excel 365)")
print("-" * 70)
print("Excel: =FILTRAR(rango, condición)")
print("Python: boolean indexing\n")

# Filtrar ventas de Laptop > $500
filtrado = df_ventas[
    (df_ventas['Producto'] == 'Laptop') & 
    (df_ventas['Total'] > 500)
][['Vendedor', 'Producto', 'Total']]

print(f"Ventas de Laptop > $500: {len(filtrado)} registros")
print(filtrado.head())

print("\n4. ORDENAR (SORT en Excel 365)")
print("-" * 70)
print("Excel: =ORDENAR(rango, columna, orden)")
print("Python: sort_values()\n")

top_ventas = df_ventas.nlargest(5, 'Total')[['Fecha', 'Vendedor', 'Producto', 'Total']]
print("Top 5 ventas:")
print(top_ventas)

print("\n5. ÚNICOS (UNIQUE en Excel 365)")
print("-" * 70)
print("Excel: =UNICOS(rango)")
print("Python: unique() o drop_duplicates()\n")

productos_unicos = df_ventas['Producto'].unique()
print(f"Productos únicos: {list(productos_unicos)}")

vendedores_unicos = df_ventas['Vendedor'].unique()
print(f"Vendedores únicos: {list(vendedores_unicos)}")

print("\n6. SECUENCIA (SEQUENCE en Excel 365)")
print("-" * 70)
print("Excel: =SECUENCIA(filas, columnas, inicio, incremento)")
print("Python: np.arange() o range()\n")

secuencia = np.arange(1, 11)
print(f"Secuencia 1-10: {secuencia}")

secuencia_par = np.arange(2, 21, 2)
print(f"Números pares 2-20: {secuencia_par}")

print("\n7. OPERACIONES MATRICIALES AVANZADAS")
print("-" * 70)
print("Excel: MMULT (multiplicación matricial)")
print("Python: np.dot() o @\n")

# Multiplicación matricial
matriz_a = np.array([[1, 2], [3, 4]])
matriz_b = np.array([[5, 6], [7, 8]])

resultado = np.dot(matriz_a, matriz_b)
print("Matriz A:")
print(matriz_a)
print("\nMatriz B:")
print(matriz_b)
print("\nA × B:")
print(resultado)

print("\n8. APLICAR FUNCIÓN A RANGO")
print("-" * 70)
print("Excel: MAP (Excel 365)")
print("Python: apply() o vectorización\n")

# Aplicar función personalizada
def clasificar_venta(total):
    if total > 1000:
        return 'Premium'
    elif total > 500:
        return 'Alta'
    else:
        return 'Normal'

df_ventas['Categoria_Venta'] = df_ventas['Total'].apply(clasificar_venta)
print(df_ventas['Categoria_Venta'].value_counts())

print("\n💡 FÓRMULAS MATRICIALES MODERNAS (Excel 365):")
print("  • FILTRAR: filtrar datos con condiciones")
print("  • ORDENAR: ordenar rangos")
print("  • UNICOS: valores únicos")
print("  • SECUENCIA: generar secuencias")
print("  • ORDENARPOR: ordenar por otra columna")
print("  • BUSCARX: BUSCARV mejorado")
print("\n  Python ofrece todas estas funcionalidades y más con Pandas/NumPy")

                         FÓRMULAS MATRICIALES                         

1. SUMA DE PRODUCTOS (SUMPRODUCT)
----------------------------------------------------------------------
Excel: =SUMAPRODUCTO(rango1, rango2)
Python: (serie1 * serie2).sum()

Total de ventas (SUMAPRODUCTO): $168,318.96
Verificación con columna Total: $168,318.98

2. OPERACIONES MATRICIALES ELEMENTO A ELEMENTO
----------------------------------------------------------------------
Excel: Fórmula matricial con Ctrl+Shift+Enter
Python: operaciones vectorizadas

Precios: [100 200 150 300]
Cantidades: [2 3 1 4]
Descuentos: [0.1  0.15 0.05 0.2 ]
Totales: [180.  510.  142.5 960. ]
Suma total: $1,792.50

3. FILTRAR CON CONDICIONES (FILTER en Excel 365)
----------------------------------------------------------------------
Excel: =FILTRAR(rango, condición)
Python: boolean indexing

Ventas de Laptop > $500: 21 registros
   Vendedor Producto    Total
1     Pedro   Laptop  4960.52
4     Pedro   Laptop  1087.64
10     Luis   Lap

## Validación de Datos y Formato Condicional

### Que es?
La validacion de datos controla entradas permitidas y el formato condicional resalta patrones visualmente.

### Para que sirve?
Mejora calidad de captura y facilita deteccion de valores relevantes o anomalias.

### Como se usa?
1. Definir reglas de entrada (listas, rangos, tipos)
2. Configurar reglas visuales por umbral o escala
3. Revisar hoja final como control de calidad

In [13]:
print("=" * 70)
print("VALIDACIÓN DE DATOS Y FORMATO CONDICIONAL".center(70))
print("=" * 70)

# Crear archivo con validación y formato condicional
print("\nCreando archivo Excel con validación y formato...")

from openpyxl.formatting.rule import ColorScaleRule, CellIsRule, IconSetRule
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.worksheet.datavalidation import DataValidation

wb = Workbook()
ws = wb.active
ws.title = "Datos con Formato"

# Escribir datos
headers = ['Vendedor', 'Producto', 'Región', 'Total', 'Categoría']
ws.append(headers)

# Estilo para encabezados
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

for col_num, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_num)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# Agregar datos de muestra
sample_data = df_ventas[['Vendedor', 'Producto', 'Región', 'Total', 'Segmento']].head(20)
for r_idx, row in enumerate(sample_data.values, 2):
    for c_idx, value in enumerate(row, 1):
        ws.cell(row=r_idx, column=c_idx, value=value)

# 1. VALIDACIÓN DE DATOS - Lista desplegable
print("\n1. VALIDACIÓN: Lista desplegable de vendedores")
print("-" * 70)

vendedores_lista = ','.join(df_ventas['Vendedor'].unique())
dv_vendedor = DataValidation(type="list", formula1=f'"{vendedores_lista}"', allow_blank=False)
dv_vendedor.error = 'Seleccione un vendedor válido'
dv_vendedor.errorTitle = 'Entrada no válida'
ws.add_data_validation(dv_vendedor)
dv_vendedor.add(f'A2:A100')
print("✓ Validación agregada en columna Vendedor (A)")

# Validación de productos
productos_lista = ','.join(df_ventas['Producto'].unique())
dv_producto = DataValidation(type="list", formula1=f'"{productos_lista}"')
ws.add_data_validation(dv_producto)
dv_producto.add(f'B2:B100')
print("✓ Validación agregada en columna Producto (B)")

# 2. FORMATO CONDICIONAL - Escala de color
print("\n2. FORMATO CONDICIONAL: Escala de color en Total")
print("-" * 70)

# Escala de color (rojo-amarillo-verde)
color_scale = ColorScaleRule(
    start_type='min',
    start_color='F8696B',
    mid_type='percentile',
    mid_value=50,
    mid_color='FFEB84',
    end_type='max',
    end_color='63BE7B'
)
ws.conditional_formatting.add('D2:D100', color_scale)
print("✓ Escala de color aplicada en columna Total (D)")

# 3. FORMATO CONDICIONAL - Reglas basadas en valores
print("\n3. FORMATO CONDICIONAL: Resaltar valores altos")
print("-" * 70)

# Resaltar valores > 500 en verde
green_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
green_font = Font(color='006100', bold=True)
green_style = DifferentialStyle(fill=green_fill, font=green_font)

rule_high = CellIsRule(
    operator='greaterThan',
    formula=['500'],
    stopIfTrue=False,
    fill=green_fill,
    font=green_font
)
ws.conditional_formatting.add('D2:D100', rule_high)
print("✓ Regla aplicada: valores > 500 resaltados en verde")

# 4. FORMATO CONDICIONAL - Iconos
print("\n4. FORMATO CONDICIONAL: Conjunto de iconos")
print("-" * 70)

# Agregar columna de rendimiento
ws['F1'] = 'Rendimiento'
ws['F1'].fill = header_fill
ws['F1'].font = header_font

# Generar valores de rendimiento
for row in range(2, 22):
    total = ws[f'D{row}'].value
    if total > 700:
        ws[f'F{row}'] = 'Alto'
    elif total > 300:
        ws[f'F{row}'] = 'Medio'
    else:
        ws[f'F{row}'] = 'Bajo'

# No todos los conjuntos de iconos están disponibles en openpyxl
# pero podemos usar reglas basadas en valores
print("✓ Categorías de rendimiento agregadas")

# 5. VALIDACIÓN NUMÉRICA
print("\n5. VALIDACIÓN: Rango numérico")
print("-" * 70)

ws['G1'] = 'Descuento %'
ws['G1'].fill = header_fill
ws['G1'].font = header_font

dv_numero = DataValidation(
    type="decimal",
    operator="between",
    formula1=0,
    formula2=100,
    allow_blank=True
)
dv_numero.prompt = 'Ingrese un valor entre 0 y 100'
dv_numero.promptTitle = 'Descuento'
dv_numero.error = 'El descuento debe estar entre 0% y 100%'
dv_numero.errorTitle = 'Error de validación'

ws.add_data_validation(dv_numero)
dv_numero.add('G2:G100')
print("✓ Validación numérica (0-100) agregada en columna G")

# Ajustar ancho de columnas
ws.column_dimensions['A'].width = 15
ws.column_dimensions['B'].width = 15
ws.column_dimensions['C'].width = 12
ws.column_dimensions['D'].width = 12
ws.column_dimensions['E'].width = 12
ws.column_dimensions['F'].width = 15
ws.column_dimensions['G'].width = 15

# Guardar
wb.save('datos_con_validacion.xlsx')
print("\n✓ Archivo 'datos_con_validacion.xlsx' creado")

print("\n📋 FUNCIONALIDADES AGREGADAS:")
print("  1. Listas desplegables en Vendedor y Producto")
print("  2. Escala de color en columna Total")
print("  3. Resaltado verde para valores > 500")
print("  4. Categorías de rendimiento")
print("  5. Validación numérica 0-100 en Descuento")

print("\n💡 EN EXCEL:")
print("  • Validación de datos: Datos → Validación de datos")
print("  • Formato condicional: Inicio → Formato condicional")
print("  • Tipos: Resaltar reglas, Barras de datos, Escalas de color, Iconos")

              VALIDACIÓN DE DATOS Y FORMATO CONDICIONAL               

Creando archivo Excel con validación y formato...

1. VALIDACIÓN: Lista desplegable de vendedores
----------------------------------------------------------------------
✓ Validación agregada en columna Vendedor (A)
✓ Validación agregada en columna Producto (B)

2. FORMATO CONDICIONAL: Escala de color en Total
----------------------------------------------------------------------
✓ Escala de color aplicada en columna Total (D)

3. FORMATO CONDICIONAL: Resaltar valores altos
----------------------------------------------------------------------
✓ Regla aplicada: valores > 500 resaltados en verde

4. FORMATO CONDICIONAL: Conjunto de iconos
----------------------------------------------------------------------
✓ Categorías de rendimiento agregadas

5. VALIDACIÓN: Rango numérico
----------------------------------------------------------------------
✓ Validación numérica (0-100) agregada en columna G

✓ Archivo 'datos_co

## Ejercicio Práctico Integrador

Crear un sistema completo de reportes en Excel.

### Que es?
Un ejercicio integrador que combina formulas, tablas dinamicas, visuales y validaciones en un solo archivo.

### Para que sirve?
Simula un entregable real para stakeholders y consolida competencias de automatizacion.

### Como se usa?
1. Construir hoja de dashboard
2. Agregar hojas de detalle y analisis
3. Aplicar formato profesional y validaciones finales

In [15]:
print("=" * 70)
print("EJERCICIO INTEGRADOR: SISTEMA DE REPORTES EN EXCEL".center(70))
print("=" * 70)

print("\nCreando sistema completo de reportes...")

# Crear archivo con múltiples hojas y funcionalidades
workbook = xlsxwriter.Workbook('sistema_reportes_ventas.xlsx')

# FORMATOS
title_format = workbook.add_format({
    'bold': True,
    'font_size': 16,
    'bg_color': '#203864',
    'font_color': 'white',
    'align': 'center',
    'valign': 'vcenter',
    'border': 1
})

header_format = workbook.add_format({
    'bold': True,
    'bg_color': '#366092',
    'font_color': 'white',
    'align': 'center',
    'border': 1
})

currency_format = workbook.add_format({'num_format': '$#,##0.00', 'border': 1})
percent_format = workbook.add_format({'num_format': '0.00%', 'border': 1})
number_format = workbook.add_format({'num_format': '#,##0', 'border': 1})
date_format = workbook.add_format({'num_format': 'dd/mm/yyyy', 'border': 1})

# KPI formats
kpi_title = workbook.add_format({
    'bold': True,
    'font_size': 11,
    'bg_color': '#D9E1F2',
    'align': 'center',
    'border': 2
})

kpi_value = workbook.add_format({
    'bold': True,
    'font_size': 14,
    'align': 'center',
    'border': 2,
    'num_format': '$#,##0.00'
})

# ==============================================================
# HOJA 1: DASHBOARD EJECUTIVO
# ==============================================================
print("\n📊 Creando Hoja 1: Dashboard Ejecutivo...")
ws_dashboard = workbook.add_worksheet('Dashboard Ejecutivo')
ws_dashboard.set_column('A:H', 15)

# Título
ws_dashboard.merge_range('A1:H1', '📊 DASHBOARD EJECUTIVO DE VENTAS', title_format)

# KPIs principales
row = 3
kpis_data = [
    ('Ventas Totales', df_ventas['Total'].sum()),
    ('Ticket Promedio', df_ventas['Total'].mean()),
    ('Num. Ventas', len(df_ventas)),
    ('Comisión Total', df_ventas['Total'].sum() * 0.05)
]

col = 0
for kpi_name, kpi_val in kpis_data:
    ws_dashboard.write(row, col, kpi_name, kpi_title)
    ws_dashboard.write(row + 1, col, kpi_val, kpi_value)
    col += 2

# Top 5 productos
row = 7
ws_dashboard.merge_range(row, 0, row, 2, 'Top 5 Productos por Ventas', header_format)
top_productos = df_ventas.groupby('Producto')['Total'].sum().sort_values(ascending=False).head(5)

row += 1
ws_dashboard.write(row, 0, 'Producto', header_format)
ws_dashboard.write(row, 1, 'Ventas', header_format)
ws_dashboard.write(row, 2, '% del Total', header_format)

total_general = df_ventas['Total'].sum()
row += 1
for producto, ventas in top_productos.items():
    ws_dashboard.write(row, 0, producto)
    ws_dashboard.write(row, 1, ventas, currency_format)
    ws_dashboard.write(row, 2, ventas / total_general, percent_format)
    row += 1

# Gráfico de productos
chart_productos = workbook.add_chart({'type': 'bar'})
chart_productos.add_series({
    'name': 'Ventas',
    'categories': ['Dashboard Ejecutivo', 9, 0, 8 + len(top_productos), 0],
    'values': ['Dashboard Ejecutivo', 9, 1, 8 + len(top_productos), 1],
    'fill': {'color': '#4472C4'},
})
chart_productos.set_title({'name': 'Top 5 Productos'})
chart_productos.set_x_axis({'name': 'Ventas ($)'})
chart_productos.set_y_axis({'name': 'Producto'})
ws_dashboard.insert_chart('E8', chart_productos, {'x_scale': 1.5, 'y_scale': 1.3})

# Ventas por región
row = 16
ws_dashboard.merge_range(row, 0, row, 1, 'Ventas por Región', header_format)
ventas_region = df_ventas.groupby('Región')['Total'].sum()

row += 1
ws_dashboard.write(row, 0, 'Región', header_format)
ws_dashboard.write(row, 1, 'Ventas', header_format)

row += 1
for region, ventas in ventas_region.items():
    ws_dashboard.write(row, 0, region)
    ws_dashboard.write(row, 1, ventas, currency_format)
    row += 1

# Gráfico de pastel por región
chart_region = workbook.add_chart({'type': 'pie'})
chart_region.add_series({
    'name': 'Ventas por Región',
    'categories': ['Dashboard Ejecutivo', 18, 0, 17 + len(ventas_region), 0],
    'values': ['Dashboard Ejecutivo', 18, 1, 17 + len(ventas_region), 1],
    'data_labels': {'percentage': True, 'category': True},
})
chart_region.set_title({'name': 'Distribución por Región'})
ws_dashboard.insert_chart('E17', chart_region, {'x_scale': 1.3, 'y_scale': 1.3})

# ==============================================================
# HOJA 2: DATOS DETALLADOS
# ==============================================================
print("📋 Creando Hoja 2: Datos Detallados...")
ws_datos = workbook.add_worksheet('Datos Detallados')

# Escribir DataFrame completo
ws_datos.merge_range('A1:K1', 'REGISTRO COMPLETO DE VENTAS', title_format)

# Headers
columnas = list(df_ventas.columns)
for col_num, col_name in enumerate(columnas):
    ws_datos.write(2, col_num, col_name, header_format)

# Datos
for row_num, row_data in enumerate(df_ventas.values, 3):
    for col_num, cell_value in enumerate(row_data):
        if columnas[col_num] == 'Fecha':
            ws_datos.write_datetime(row_num, col_num, cell_value, date_format)
        elif columnas[col_num] in ['Precio_Unitario', 'Total']:
            ws_datos.write_number(row_num, col_num, cell_value, currency_format)
        elif columnas[col_num] == 'Descuento':
            ws_datos.write_number(row_num, col_num, cell_value, percent_format)
        elif columnas[col_num] == 'Cantidad':
            ws_datos.write_number(row_num, col_num, cell_value, number_format)
        else:
            ws_datos.write(row_num, col_num, cell_value)

# Autofilter
ws_datos.autofilter(2, 0, len(df_ventas) + 2, len(columnas) - 1)

# Ajustar columnas
ws_datos.set_column('A:A', 12)  # Fecha
ws_datos.set_column('B:B', 12)  # Vendedor
ws_datos.set_column('C:C', 15)  # Producto
ws_datos.set_column('D:K', 12)  # Resto

# ==============================================================
# HOJA 3: ANÁLISIS POR VENDEDOR
# ==============================================================
print("👤 Creando Hoja 3: Análisis por Vendedor...")
ws_vendedor = workbook.add_worksheet('Análisis Vendedor')

ws_vendedor.merge_range('A1:F1', 'ANÁLISIS POR VENDEDOR', title_format)

# Tabla resumen
analisis_vendedor = df_ventas.groupby('Vendedor').agg({
    'Total': ['sum', 'mean', 'count'],
    'Cantidad': 'sum'
}).round(2)

analisis_vendedor.columns = ['Total_Ventas', 'Promedio_Venta', 'Num_Ventas', 'Unidades_Vendidas']
analisis_vendedor = analisis_vendedor.sort_values('Total_Ventas', ascending=False).reset_index()
analisis_vendedor['Comision'] = analisis_vendedor['Total_Ventas'] * 0.05

# Headers
headers_vendedor = ['Vendedor', 'Total Ventas', 'Promedio', 'Num. Ventas', 'Unidades', 'Comisión']
for col_num, header in enumerate(headers_vendedor):
    ws_vendedor.write(2, col_num, header, header_format)

# Datos
for row_num, row_data in enumerate(analisis_vendedor.values, 3):
    ws_vendedor.write(row_num, 0, row_data[0])
    ws_vendedor.write(row_num, 1, row_data[1], currency_format)
    ws_vendedor.write(row_num, 2, row_data[2], currency_format)
    ws_vendedor.write(row_num, 3, row_data[3], number_format)
    ws_vendedor.write(row_num, 4, row_data[4], number_format)
    ws_vendedor.write(row_num, 5, row_data[5], currency_format)

# Gráfico de vendedores
chart_vendedor = workbook.add_chart({'type': 'column'})
chart_vendedor.add_series({
    'name': 'Ventas Totales',
    'categories': ['Análisis Vendedor', 3, 0, 2 + len(analisis_vendedor), 0],
    'values': ['Análisis Vendedor', 3, 1, 2 + len(analisis_vendedor), 1],
    'fill': {'color': '#70AD47'},
})
chart_vendedor.set_title({'name': 'Desempeño por Vendedor'})
chart_vendedor.set_x_axis({'name': 'Vendedor'})
chart_vendedor.set_y_axis({'name': 'Ventas ($)'})
ws_vendedor.insert_chart('A10', chart_vendedor, {'x_scale': 2, 'y_scale': 1.5})

ws_vendedor.set_column('A:F', 15)

# ==============================================================
# HOJA 4: TABLA DINÁMICA
# ==============================================================
print("📊 Creando Hoja 4: Tabla Dinámica...")
ws_pivot = workbook.add_worksheet('Tabla Dinámica')

ws_pivot.merge_range('A1:F1', 'TABLA DINÁMICA: VENTAS POR PRODUCTO Y REGIÓN', title_format)

# Crear pivot table
pivot = pd.pivot_table(
    df_ventas,
    values='Total',
    index='Producto',
    columns='Región',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
).round(2)

# Escribir pivot
ws_pivot.write(2, 0, 'Producto', header_format)
for col_num, col_name in enumerate(pivot.columns, 1):
    ws_pivot.write(2, col_num, str(col_name), header_format)

row = 3
for producto, datos in pivot.iterrows():
    ws_pivot.write(row, 0, str(producto), header_format if producto == 'TOTAL' else None)
    for col_num, valor in enumerate(datos, 1):
        formato = currency_format if producto == 'TOTAL' else currency_format
        ws_pivot.write(row, col_num, valor, formato)
    row += 1

ws_pivot.set_column('A:F', 15)

# Cerrar workbook
workbook.close()

print("\n✅ SISTEMA DE REPORTES COMPLETADO")
print("=" * 70)
print("\n📁 Archivo creado: sistema_reportes_ventas.xlsx")
print("\n📑 HOJAS INCLUIDAS:")
print("  1. Dashboard Ejecutivo:")
print("     • KPIs principales")
print("     • Top 5 productos con gráfico")
print("     • Distribución por región con gráfico de pastel")
print("\n  2. Datos Detallados:")
print("     • Registro completo con formato")
print("     • Filtros automáticos")
print("     • Formato condicional")
print("\n  3. Análisis por Vendedor:")
print("     • Métricas por vendedor")
print("     • Cálculo de comisiones")
print("     • Gráfico comparativo")
print("\n  4. Tabla Dinámica:")
print("     • Ventas por Producto y Región")
print("     • Totales y subtotales")

print("\n🎯 FUNCIONALIDADES IMPLEMENTADAS:")
print("  ✓ Fórmulas avanzadas (SUMAR.SI, SI, BUSCARV)")
print("  ✓ Tablas dinámicas")
print("  ✓ Gráficos múltiples (barras, columnas, pastel)")
print("  ✓ Formato profesional")
print("  ✓ KPIs destacados")
print("  ✓ Filtros automáticos")
print("  ✓ Múltiples hojas interrelacionadas")

print("\n💡 Este sistema puede abrirse directamente en Excel")
print("   y contiene todas las funcionalidades de un reporte profesional.")

          EJERCICIO INTEGRADOR: SISTEMA DE REPORTES EN EXCEL          

Creando sistema completo de reportes...

📊 Creando Hoja 1: Dashboard Ejecutivo...
📋 Creando Hoja 2: Datos Detallados...
👤 Creando Hoja 3: Análisis por Vendedor...
📊 Creando Hoja 4: Tabla Dinámica...

✅ SISTEMA DE REPORTES COMPLETADO

📁 Archivo creado: sistema_reportes_ventas.xlsx

📑 HOJAS INCLUIDAS:
  1. Dashboard Ejecutivo:
     • KPIs principales
     • Top 5 productos con gráfico
     • Distribución por región con gráfico de pastel

  2. Datos Detallados:
     • Registro completo con formato
     • Filtros automáticos
     • Formato condicional

  3. Análisis por Vendedor:
     • Métricas por vendedor
     • Cálculo de comisiones
     • Gráfico comparativo

  4. Tabla Dinámica:
     • Ventas por Producto y Región
     • Totales y subtotales

🎯 FUNCIONALIDADES IMPLEMENTADAS:
  ✓ Fórmulas avanzadas (SUMAR.SI, SI, BUSCARV)
  ✓ Tablas dinámicas
  ✓ Gráficos múltiples (barras, columnas, pastel)
  ✓ Formato profesiona

## Resumen

### Conceptos clave aprendidos:

**Funciones Avanzadas:**
- BUSCARV: `df.merge()` o `Series.map()`
- SI: `np.where()` o `np.select()`
- SUMAR.SI: `df.groupby()` o filtrado + `sum()`
- CONTAR.SI: `value_counts()` o filtrado booleano

**Tablas Dinámicas:**
- `pd.pivot_table()` para agregaciones
- `pd.crosstab()` para frecuencias
- Múltiples dimensiones y funciones

**Gráficos y Dashboards:**
- OpenPyXL: leer/modificar Excel existentes
- XlsxWriter: crear nuevos con gráficos avanzados
- Integración con Pandas

**Power Query:**
- Transformaciones con Pandas
- Filtrado, agrupación, combinación
- Pivot/Unpivot con `melt()`

**Fórmulas Matriciales:**
- SUMAPRODUCTO, FILTRAR, ORDENAR, UNICOS
- Operaciones vectorizadas con NumPy
- Funciones modernas de Excel 365

**Validación y Formato:**
- Listas desplegables
- Formato condicional (escalas, reglas, iconos)
- Validación numérica y de texto